# Семинар 7: Градиентный бустинг

Данный ноутбук носит обзорный характер, для изучения более сложных математических основ, рекомендуется самостоятельно пройти по предложенным ссылкам :)

In [ ]:
from setup import *

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, fetch_california_housing, load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, mean_squared_error

## Краткий повтор

Бустинг — это ансамблевый метод, который последовательно строит композицию алгоритмов. Каждый следующий алгоритм исправляет ошибки предыдущих.

### Простейший алгоритм градиентного бустинга

1. **Инициализация** композиции оптимальным константным значением:
   $$a_0 = \arg\min_{c \in \mathbb{R}} \sum_{i=1}^n \mathcal{L}(c, y_i)$$
   *Где $\mathcal{L}$ — лосс (например, MSE для регрессии или LogLoss для классификации).*

2. Для всех $k = 1, \dots, T$:
    * **Вычислить градиент** (псевдо-остатки) ошибки предыдущей композиции:
      $$g_{i}^{k} = -\left[\frac{\partial \mathcal{L}(a_k, x_i, y_i)}{\partial a_k(x_i)}\right]_{i=1}^N$$
    * **Обучить базовую модель** $b_k(x)$ на полученные остатки: $\{(x_i, g_{i}^{k})\}$.
    * **Найти оптимальный коэффициент** $\eta_k$ перед базовым алгоритмом:
      $$\eta_k = \arg\min_\eta \sum_{i=1}^N \mathcal{L}(a_{k-1}(x_i) + \eta b_k(x_i), y_i)$$
    * **Обновить композицию**: $a_k(x) = a_{k-1}(x) + \eta_k b_k(x)$.

В результате получаем итоговый алгоритм: $a(x) = a_T(x) = \sum_{t=1}^T \eta_tb_t(x)$.

*`Идея проста`: мы спускаемся по градиенту функционала качества в пространстве функций, а роль "шага" выполняет новый базовый алгоритм.*

![alt](../data/6.webp)

Давайте теперь поговорим о конкретных реализациях бустинга, коих в ML существует много..

Есть даже отдельная [статья](https://arxiv.org/pdf/2307.07771v1) с их сравнением

![alt](../data/8.png)

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor

## 2. AdaBoost (Adaptive Boosting)

На лекции обсуждалась реализация градиентного бустинга для задачи регрессии. Несложно определить её для классификации

В качестве лосса будем использовать `logloss`, будем аппроксимировать margin, а сам результат работы модели интерпретировать вероятностно (см. лекцию про логистическую регрессию)

Находим нужные производные функции потерь

$$\mathcal{L}(a_k, x_i, y_i) = y_{i}\ln \sigma(a_k (x_i)) + (1 - y_{i})\ln(1 - \sigma(a_k ( x_i)))$$

$$g_{i}^{k} = -\Big[\frac{\partial \mathcal{L}(a_k, x_i, y_i)}{\partial a_k(x_i)}\Big]_{i=1}^N$$

А теперь находим $a_{k+1}$ через решение регрессии

Результат же мы также будем получать через вероятность. 

Пусть $a_T(x)$ - итоговый алгоритм, тогда:

$$a(x) = \begin{cases}
1, & \sigma(a_T(x)) > 0.5 \\ 
0, & \sigma(a_T(x)) < 0.5 \\ 
\end{cases}$$

### Постановка задачи
* $(X, y)_{N}$ - тренировочная выборка длины $N$, где $y_i \in \{-1, 1\}$ ($-1$ вместо $0$ - удобнее для алгоритма)
* $Q(a, X, y)$ - **количество ошибок**


$$Q(a, X, y) = \sum_{i=1}^{N}\mathcal{I}(y_i \neq a(x)) = \sum_{i=1}^{N}[y_ia(x_i) < 0] =\sum_{i=1}^{N}[M < 0]$$

$[M < 0]$ - пороговая функция, которая равная $1$, если условие выполняется и $0$, если не выполняется

Таким образом получим, что если реальный ответ и ответ алгоритма не совпадают - значение $M$ будет отрицательно и пороговая функция обратится в 0, иначе в 1.

Если мы ищем алгоритм в виде бустинга, то получим

$$a(x) = \sum_{t = 1}^T \eta_t b_t(x)$$

$$Q_T(X, y) = \sum_{i=1}^{N}\Big[y_i\sum_{t = 1}^T \alpha_t b_t(x_i) < 0\Big]$$


Используя различные аппроксимации для пороговой функции потерь $[M < 0]$, будем получать различные виды бустинга.
Если аппроксимация выглядит вот так: $Exp(-M)$, то полученная реализация бустинга называется - `AdaBoost`

### Суть алгоритма
В отличие от градиентного бустинга, AdaBoost фокусируется на *перевзвешивании объектов*. Ошибочно классифицированные объекты на текущем шаге получают больший вес на следующем.

**Формула обновления весов (для классификации):**

1. Инициализируем веса объектов: $w_i^{(1)} = \frac{1}{N}$.
2. Для каждого шага $t$:
    - Обучаем слабый классификатор $b_t$ с учётом весов $w^{(t)}$.
    - Вычисляем ошибку: $s_t = \frac{\sum_{i=1}^N w_i^{(t)} \cdot [y_i \neq b_t(x_i)]}{\sum_{i=1}^N w_i^{(t)}}$.
    - Вычисляем "вес голоса" классификатора: $\alpha_t = \frac{1}{2} \ln\left(\frac{1-s_t}{s_t}\right)$.
    - Обновляем веса объектов: $w_i^{(t+1)} = w_i^{(t)} \cdot \exp(-\alpha_t y_i b_t(x_i))$.

Тем, кому нужны математические выкладки [тык сюда](https://cmp.felk.cvut.cz/~sochmj1/adaboost_talk.pdf) 

`Примечание`: в оригинале в качестве слабых классификаторов используются **пни решений** (деревья с глубиной 1)

### Пример на "пнях решения" (Decision Stumps)

In [ ]:
# Загружаем данные
data = load_iris()
X, y = data.data, data.target
# Для бинарной классификации возьмем два класса
X = X[y != 2]
y = y[y != 2]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Создаем AdaBoost с "пнями" (max_depth=1)
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=50,
    algorithm='SAMME',
    random_state=42
)

ada_model.fit(X_train, y_train)
y_pred = ada_model.predict(X_test)

print(f"Accuracy на тесте: {accuracy_score(y_test, y_pred):.4f}")
print(f"Количество слабых классификаторов: {len(ada_model.estimators_)}")

**Когда использовать AdaBoost:**
- Когда важна интерпретируемость (комбинация простых "пней" часто легче объяснима).
- Для бинарной классификации, когда нет сильных выбросов (AdaBoost чувствителен к шуму).
- Если нужно быстрое решение без сложной настройки гиперпараметров.

## 3. XGBoost (Extreme Gradient Boosting)

Теперь рассмотрим первый алгоритм, который находит вне пакета `sklearn`.
Его создавала сторонняя команда с большим числом прикольных фич.

[Xgboost, Регуляризация](https://xgboost.readthedocs.io/en/latest/tutorials/model.html?highlight=boosting%20trees)

Проблема многих алгоритмов построения деревьев в том, что в них не уделяется должного внимания **регуляризации**. 
В классическом градиентном бустинге применяется такие меры:
- ограничение на структуру дерева: максимальная глубина (max_depth), минимальное число объектов в листе (min_samples_leaf)
- контролирование темпа обучения (learning_rate)
- увеличение "непохожести" деревьев за счет рандомизации, как в случайном лесе

[Xgboost](https://github.com/dmlc/xgboost) использует еще больше параметров для регуляризации базовых деревьев.

Бустинг строит композицию из $K$ базовых алгоритмов $b_k$:

$$ \hat{y}_i = \hat{y}_i^{K} = \sum_{k=1}^{K} b_k(x_i) = \hat{y}_i^{\left(K - 1\right)} + b_K(x_i), $$

С учетом регуляризации XGBoost минимизирует следующий функционал:

$$ Obj = \sum_{i=1}^N \mathcal{L}(y_i, \hat{y}_i ) + \sum_{k=1}^{K} \Omega(b_k),$$

где
 - $N$ — размер обучающей выборки;
 - $x_i, y_i, \hat{y}_i$ — i-ый объект, правильный ответ и предсказание модели для него;
 - $\hat{y}_i^{t}$ — предсказание композиции из $t$ уже обученных базовых алгоритмов для i-го объекта;
 - $\Omega$ — регуляризатор;
 - $\mathcal{L}(y_i, \hat{y}_i)$ — функция потерь.


Функционал, оптимизируемый на $t$-ой итерации:

$$ Obj^{(t)} = \sum_{i=1}^N \mathcal{L}\left(y_i, \hat{y}_i^{(t-1)} + b_t(x_i)\right) + \Omega(b_t).$$

В случае бустинга над **решающими деревьями** регуляризатор имеет следующий вид:

$$ \Omega(b_t) = \gamma T + \frac{1}{2}\lambda\sum_{j=1}^{T}w_j^2 + \alpha\sum_{j=1}^{T}|w_j|,$$

где 
 - $T$ — **количество листьев в дереве**;
 - $w_j$ — **веса в листьях дерева** (вероятность попадания в лист по тренировочной выборке);
 - $\lambda, \alpha, \gamma$ — **гиперпараметры**

In [ ]:
# Загружаем датасет California Housing
housing = fetch_california_housing()
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Создаем и обучаем модель
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    reg_alpha=0.1,  # L1 регуляризация
    reg_lambda=1.0, # L2 регуляризация
    random_state=42
)

xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE на тесте: {rmse:.4f}")

# Вывод feature importances (по умолчанию F-score)
importance = pd.DataFrame({
    'feature': housing.feature_names,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)
print("\nВажность признаков:")
print(importance)

**Когда использовать XGBoost:**
- Для табличных данных, где важна точность.
- Когда в данных есть пропуски (алгоритм обрабатывает их нативно).
- Если требуется контроль переобучения через мощную регуляризацию.
- Для соревнований (Kaggle) — стандарт индустрии.

## 4. LightGBM (Light Gradient Boosting Machine)


`LightGBM` (Light Gradient Boosting Machine) — это библиотека от Microsoft, реализующая алгоритм градиентного бустинга на деревьях решений. Она ценится за экстремальную скорость, эффективность использования памяти и высокую точность на больших данных.

### Особенности алгоритма:

* `Leaf-wise рост деревьев`: В отличие от большинства библиотек, которые строят деревья слой за слоем (level-wise), LightGBM выбирает лист с максимальным уменьшением ошибки и разбивает его. Это позволяет быстрее снижать лосс, хотя требует контроля переобучения.
* `GOSS (Gradient-based One-Side Sampling)`: Алгоритм оставляет объекты с большими градиентами (на которых модель ошибается сильнее всего) и случайным образом отбирает часть объектов с маленькими градиентами, что ускоряет обучение без потери точности.
* `EFB (Exclusive Feature Bundling)`: Метод объединения разреженных (sparse) признаков, что уменьшает их количество и ускоряет процесс.
* `Поддержка гистограмм`: Непрерывные признаки разбиваются на «корзины» (bins), что на порядки ускоряет поиск лучшего разделения.

### Демонстрация скорости на большом объеме данных

In [ ]:
# 1. Загрузка данных
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

# 2. Инициализация классификатора
# Параметр n_estimators — кол-во деревьев, learning_rate — шаг обучения
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

# 3. Обучение
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='binary_logloss'
)

# 4. Предсказание и оценка
y_pred = model.predict(X_test)
print(f"Точность модели: {accuracy_score(y_test, y_pred):.4f}")

**Когда использовать LightGBM:**
- Для очень (очень-очень) больших датасетов (миллионы строк).  Если XGBoost работает слишком медленно, LightGBM обычно справляется быстрее.
- Когда критична скорость обучения и инференса.
- Для задач с высокой размерностью (много признаков).
- А ещё: `Категориальные признаки`... В библиотеку встроена поддержка категорий (не нужно делать One-Hot Encoding вручную).

## 5. CatBoost (Categorical Boosting)

[Подробный разбор](https://habr.com/ru/companies/otus/articles/778714/) catboost для интересующихся

### Работа с категориальными фичами "из коробки"

CatBoost (сокращение от Categorical Boosting) — это открытая библиотека градиентного бустинга, разработанная в Яндексе. Она специально заточена под работу с табличными данными, где много текстовых (категориальных) признаков.
Если LightGBM берет скоростью, а XGBoost — точностью на числах, то CatBoost — это «король категорий».

### Пример, где мы не делаем One-Hot Encoding вручную

In [ ]:
rng = np.random.RandomState(31337)

boston = load_diabetes()
y = boston['target']
X = boston['data']

kf = KFold(n_splits=3, shuffle=True, random_state=rng)

X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.25)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5)

In [ ]:
cb = CatBoostRegressor(silent=True, eval_metric="MAE", custom_metric=["MAPE"])

In [ ]:
# Также в пакет включена крутая интерактивная визуализация

cb.fit(X_train, y_train, eval_set=[(X_val , y_val ), (X_test, y_test)], plot=True)

### Как он это делает? (Алгоритм TS)

Вместо того чтобы просто заменить «Москва» на 1, а «Питер» на 2, CatBoost использует **Target Statistics (TS)**.

1. **Случайная перестановка:** Чтобы модель не «зазубрила» данные (не переобучилась), CatBoost перемешивает строки.
2. **Вычисление значения:** Для каждой строки он считает среднее значение целевой переменной (Target) только по тем строкам того же города, что стоят выше неё.

3. **Комбинации признаков:** Это киллер-фича. CatBoost на лету пробует объединять категории (например, «Город» + «Тип товара»), создавая новые сложные признаки, что другие бустинги сами не умеют.



**Когда использовать CatBoost:**
- Если в данных много категориальных признаков (строковые типы, ID, коды).
- Когда нужно быстро получить хорошее качество без предобработки.
- Для защиты от переобучения на малых выборках (благодаря симметричным деревьям).

Давайте теперь сравним все известные нам алгоритмы GradientBoosting

In [ ]:
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=150, noise=0.25, random_state=42)
animate_boosting_comparison(X, y, frames=50)

* `Vanilla GBM (Sklearn)`: Классический Gradient Boosting строит деревья по уровням (level-wise), границы меняются плавно и предсказуемо.
* `XGBoost`: Часто дает более агрессивные, «квадратные» границы из-за продвинутой регуляризации.
* `LightGBM`: За счет роста по листьям (Leaf-wise) он быстрее всех находит сложные локальные паттерны, что на анимации выглядит как быстрое появление узких зон уверенности.
* `CatBoost`: Использует симметричные деревья, поэтому его границы обычно выглядят более «упорядоченными» и менее зашумленными на ранних этапах.

## 6. Интерпретация модели с SHAP

`SHAP` (SHapley Additive exPlanations) — это метод объяснения предсказаний любых моделей машинного обучения, основанный на теории кооперативных игр

Если обычный «feature importance» говорит, какие признаки важны в целом для всей модели, то SHAP показывает, **как именно каждый признак повлиял на каждый конкретный прогноз**.



### Основная идея (Значения Шепли)

Представьте, что признаки — это игроки в команде, а предсказание модели — это общий выигрыш. **SHAP** распределяет этот выигрыш между игроками максимально справедливо, учитывая вклад каждого в разных комбинациях с другими.

* Положительное значение SHAP: признак тянет предсказание вверх (например, увеличивает цену квартиры или вероятность болезни).
* Отрицательное значение SHAP: признак тянет предсказание вниз.
* Сумма вкладов: сумма всех SHAP-значений признаков плюс среднее предсказание по выборке всегда равна финальному ответу модели для данного объекта.

### Как это выглядит?

1. **Summary Plot (Beeswarm)**: Общая картина. Показывает распределение вкладов каждого признака. Каждая точка — это один объект данных.
    * Цвет: красный — высокое значение признака, синий — низкое.
    * Положение: справа от центра — признак увеличивает предсказание, слева — уменьшает.
2. **Waterfall / Force Plot**: Детальный разбор одного конкретного предсказания. Видно, какие именно факторы «перевесили» в ту или иную сторону.
3. **Dependence Plot**: Показывает зависимость между значением признака и его вкладом, а также взаимодействия с другими признаками.

Для моделей на основе деревьев (LGBM, XGBoost, CatBoost) существует оптимизированный TreeExplainer, который работает очень быстро.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
from sklearn.model_selection import train_test_split

# 1. Данные (сразу создаем DataFrame, чтобы не было ошибок с типами)
from sklearn.datasets import make_classification
X_raw, y = make_classification(n_samples=500, n_features=10, random_state=42)
X = pd.DataFrame(X_raw, columns=[f'Feature_{i}' for i in range(10)])

# Добавляем текстовую колонку для теста категорий
X['Category_Col'] = np.random.choice(['A', 'B', 'C'], size=500)
X['Category_Col'] = X['Category_Col'].astype('category')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Обучение
model = lgb.LGBMClassifier(n_estimators=100, verbose=-1).fit(X_train, y_train)

# 3. SHAP
# TreeExplainer — самый быстрый для деревьев
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

# ВАЖНО: Для классификации берем значения только для класса 1 (Positive)
# В новых версиях shap_values имеет размерность (объекты, признаки, классы)
if len(shap_values.shape) == 3:
    actual_shap_values = shap_values[:, :, 1]
else:
    actual_shap_values = shap_values

# 4. Визуализация
shap.initjs() # Нужно для отображения в Notebook

# Общий график важности
shap.plots.beeswarm(actual_shap_values)

# Разбор первого предсказания
shap.plots.waterfall(actual_shap_values[0])

## 7. Автоматический подбор гиперпараметров с Optuna

[Optuna](https://optuna.org/) — это современная библиотека на Python для автоматического подбора (оптимизации) гиперпараметров моделей машинного обучения. Она работает значительно эффективнее классических методов вроде GridSearchCV за счет использования интеллектуальных алгоритмов поиска.

Optuna использует алгоритм **TPE (Tree-structured Parzen Estimator)** для эффективного перебора гиперпараметров. Он автоматически отсеивает неудачные конфигурации (pruning).

### Trial

Trial (Испытание) — это одна конкретная попытка обучения модели с набором параметров.

1. Внутри одного *Trial* программа берет значения (например, `learning_rate=0.01`), обучает модель и возвращает число (результат).

2. *Study* (Исследование) — это совокупность всех триалов. Optuna анализирует историю прошлых триалов, чтобы в следующем триале предложить параметры получше.

### Define-by-Run (Динамика)
В отличие от GridSearch, где сетка параметров жесткая, в `Optuna` пространство поиска описывается прямо внутри функции через `trial.suggest_`.... Это позволяет строить условия: «Если алгоритм — `Random Forest`, подбирай глубину, а если `XGBoost` — подбирай число деревьев»



In [ ]:
import optuna
import lightgbm as lgb
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
import pandas as pd

# 1. Подготовка данных (чтобы X_train и X_test существовали)
X_raw, y = make_classification(n_samples=500, n_features=10, random_state=42)
X = pd.DataFrame(X_raw)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Функция-цель для Optuna
def objective(trial):
    param = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
    }

    model = lgb.LGBMClassifier(**param)
    model.fit(X_train, y_train)

    # ИСПОЛЬЗУЕМ X_test, так как X_val не определен
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    return accuracy

# 3. Запуск оптимизации
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print(f"Лучший результат: {study.best_value}")
print(f"Лучшие параметры: {study.best_params}")

## 8. Практическое задание

**Цель:** Закрепить навыки работы с различными библиотеками бустинга, оптимизацией гиперпараметров и интерпретацией моделей.

**Задача:**
1. **Выберите датасет** (на ваш выбор):
   - `sklearn.datasets.fetch_covtype` (классификация лесного покрова)
   - `sklearn.datasets.load_digits` (классификация рукописных цифр)

2. **Сравните 3 модели бустинга** (XGBoost, LightGBM, CatBoost), выберите подходящую метрику.

3. **Оптимизируйте лучшую модель** (по качеству) с помощью Optuna. Проведите 20-30 итераций (trial).

4. **Объясните топ-5 признаков** для модели.

In [ ]:
# Ваш код здесь